<a href="https://colab.research.google.com/github/sadikinisaac/AIML/blob/main/sgstraitsvesselmovement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# SINGAPORE STRAIT — VESSEL TRAFFIC ANIMATION
# Free basemap via OpenStreetMap + contextily | 3-min MP4 | 102 vessels
# ═══════════════════════════════════════════════════════════════════════════════

# ── Install ──────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'contextily', 'pyproj', 'numpy', 'matplotlib', 'tqdm'], check=True)
subprocess.run(['apt-get', 'install', '-q', '-y', 'ffmpeg'], check=True)

# ── Imports ───────────────────────────────────────────────────────────────────
import io, os, math, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.animation import FFMpegWriter
from pyproj import Transformer
import contextily as ctx
from tqdm.auto import tqdm
from google.colab import files

# ═══════════════════════════════════════════════════════════════════════════════
# 1. MAP EXTENT  (Web Mercator EPSG:3857)
# ═══════════════════════════════════════════════════════════════════════════════

LON_MIN, LON_MAX = 103.55, 104.35
LAT_MIN, LAT_MAX = 1.10,  1.45

transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
xmin, ymin = transformer.transform(LON_MIN, LAT_MIN)
xmax, ymax = transformer.transform(LON_MAX, LAT_MAX)

def to_merc(lons, lats):
    return transformer.transform(lons, lats)

# ═══════════════════════════════════════════════════════════════════════════════
# 2. FETCH & CACHE BASEMAP  (OpenStreetMap satellite — Esri World Imagery)
# ═══════════════════════════════════════════════════════════════════════════════

MAP_CACHE = '/tmp/sg_basemap_osm.npz'

if os.path.exists(MAP_CACHE):
    data      = np.load(MAP_CACHE)
    basemap   = data['img']
    bm_extent = list(data['extent'])
    print(f'✅  Loaded cached basemap  {basemap.shape[1]}×{basemap.shape[0]} px')
else:
    print('⬇️  Downloading basemap (Esri World Imagery via contextily)…')
    fig_tmp, ax_tmp = plt.subplots(figsize=(12, 7))
    ax_tmp.set_xlim(xmin, xmax)
    ax_tmp.set_ylim(ymin, ymax)
    ctx.add_basemap(ax_tmp, source=ctx.providers.Esri.WorldImagery, zoom=11, crs='EPSG:3857')
    # Extract the image artist
    imgs = [c for c in ax_tmp.get_children()
            if isinstance(c, matplotlib.image.AxesImage)]
    bm_img_artist = imgs[0]
    basemap   = (bm_img_artist.get_array() * 255).astype(np.uint8) \
                if bm_img_artist.get_array().max() <= 1.0 \
                else bm_img_artist.get_array().astype(np.uint8)
    bm_extent = list(bm_img_artist.get_extent())   # [x0, x1, y0, y1]
    plt.close(fig_tmp)
    np.savez_compressed(MAP_CACHE, img=basemap, extent=np.array(bm_extent))
    print(f'✅  Basemap saved  {basemap.shape[1]}×{basemap.shape[0]} px')

# ═══════════════════════════════════════════════════════════════════════════════
# 3. ANIMATION PARAMETERS
# ═══════════════════════════════════════════════════════════════════════════════

DURATION_S = 180
FPS        = 24
N_FRAMES   = DURATION_S * FPS    # 4 320
TRAIL      = FPS * 6             # 6-second wake trail
SIM_HOURS  = 12.0
SIM_START  = 6.0                 # simulation starts at 06:00 SGT

print(f'Total frames: {N_FRAMES}  ({DURATION_S}s × {FPS}fps)  |  102 vessels')

# ═══════════════════════════════════════════════════════════════════════════════
# 4. VESSEL TRACK GENERATION
# ═══════════════════════════════════════════════════════════════════════════════

rng = np.random.default_rng(42)

def make_track(lon0, lat0, lon1, lat1,
               spread_lon=0.008, spread_lat=0.006,
               speed_var=0.15, stagger=0, wobble=0.0005):
    dlon = rng.uniform(-spread_lon, spread_lon)
    dlat = rng.uniform(-spread_lat, spread_lat)
    speed = rng.normal(1.0, speed_var, N_FRAMES).clip(0.3, 2.0)
    cum   = np.cumsum(speed)
    t     = (cum - cum[0]) / (cum[-1] - cum[0])
    lon   = (lon0 + dlon) + t * (lon1 - lon0)
    lat   = (lat0 + dlat) + t * (lat1 - lat0)
    # lateral sinusoidal wobble
    phase = rng.uniform(0, 2 * np.pi)
    freq  = rng.uniform(0.3, 0.8)
    side  = np.sin(2 * np.pi * freq * t + phase) * wobble
    dx, dy = lon1 - lon0, lat1 - lat0
    norm = math.hypot(dx, dy) or 1
    lon += side * (-dy / norm)
    lat += side * ( dx / norm)
    if stagger > 0:
        lon = np.roll(lon, stagger); lon[:stagger] = np.nan
        lat = np.roll(lat, stagger); lat[:stagger] = np.nan
    return lon, lat

vessels = []

# Eastbound TSS (30)
for _ in range(30):
    stag = int(rng.integers(0, N_FRAMES // 2))
    l, a = make_track(103.58, 1.168, 104.28, 1.178,
                      spread_lon=0.0, spread_lat=0.012, stagger=stag)
    vessels.append({'type': 'eastbound', 'lon': l, 'lat': a})

# Westbound TSS (30)
for _ in range(30):
    stag = int(rng.integers(0, N_FRAMES // 2))
    l, a = make_track(104.28, 1.210, 103.58, 1.200,
                      spread_lon=0.0, spread_lat=0.012, stagger=stag)
    vessels.append({'type': 'westbound', 'lon': l, 'lat': a})

# SW / Philip Channel crossing (10)
for _ in range(10):
    stag = int(rng.integers(0, N_FRAMES // 2))
    l, a = make_track(103.58, 1.230, 103.88, 1.190,
                      spread_lon=0.01, spread_lat=0.008, stagger=stag)
    vessels.append({'type': 'crossing', 'lon': l, 'lat': a})

# Northbound – Changi / Johor Strait (6)
for _ in range(6):
    stag = int(rng.integers(0, N_FRAMES // 2))
    l, a = make_track(103.95 + rng.uniform(-0.04, 0.04), 1.155,
                      103.93 + rng.uniform(-0.04, 0.04), 1.330,
                      spread_lon=0.015, spread_lat=0.0, stagger=stag)
    vessels.append({'type': 'northbound', 'lon': l, 'lat': a})

# Southbound – Jurong / Keppel (6)
for _ in range(6):
    stag = int(rng.integers(0, N_FRAMES // 2))
    l, a = make_track(103.73 + rng.uniform(-0.04, 0.04), 1.330,
                      103.71 + rng.uniform(-0.04, 0.04), 1.155,
                      spread_lon=0.015, spread_lat=0.0, stagger=stag)
    vessels.append({'type': 'southbound', 'lon': l, 'lat': a})

# Anchored – Eastern Anchorage (12)
for _ in range(12):
    alon = rng.uniform(104.00, 104.20)
    alat = rng.uniform(1.220, 1.270)
    dl = np.cumsum(rng.normal(0, 0.00005, N_FRAMES))
    da = np.cumsum(rng.normal(0, 0.00003, N_FRAMES))
    vessels.append({'type': 'anchored', 'lon': alon + dl, 'lat': alat + da})

# Anchored – Western Anchorage (8)
for _ in range(8):
    alon = rng.uniform(103.58, 103.80)
    alat = rng.uniform(1.155, 1.210)
    dl = np.cumsum(rng.normal(0, 0.00005, N_FRAMES))
    da = np.cumsum(rng.normal(0, 0.00003, N_FRAMES))
    vessels.append({'type': 'anchored', 'lon': alon + dl, 'lat': alat + da})

print(f'Vessels spawned: {len(vessels)}')

# ═══════════════════════════════════════════════════════════════════════════════
# 5. PRE-COMPUTE MERCATOR COORDS  (skip repeated transformer calls per frame)
# ═══════════════════════════════════════════════════════════════════════════════

print('⚙️  Pre-computing Mercator coordinates…')
merc_coords = []
for v in vessels:
    lons, lats = v['lon'], v['lat']
    valid = ~np.isnan(lons)
    xs = np.full(N_FRAMES, np.nan)
    ys = np.full(N_FRAMES, np.nan)
    if valid.any():
        xs[valid], ys[valid] = to_merc(lons[valid], lats[valid])
    merc_coords.append((xs, ys))
print('✅  Done')

# ═══════════════════════════════════════════════════════════════════════════════
# 6. STYLE MAP
# ═══════════════════════════════════════════════════════════════════════════════

STYLE = {
    'eastbound':  {'color': '#00cfff', 'size': 28, 'marker': '>', 'zorder': 5},
    'westbound':  {'color': '#ff6b35', 'size': 28, 'marker': '<', 'zorder': 5},
    'crossing':   {'color': '#ffe066', 'size': 22, 'marker': '^', 'zorder': 4},
    'northbound': {'color': '#a0ff80', 'size': 22, 'marker': '^', 'zorder': 4},
    'southbound': {'color': '#ff80d5', 'size': 22, 'marker': 'v', 'zorder': 4},
    'anchored':   {'color': '#ffdd00', 'size': 20, 'marker': 's', 'zorder': 3},
}

# ═══════════════════════════════════════════════════════════════════════════════
# 7. FIGURE SETUP
# ═══════════════════════════════════════════════════════════════════════════════

DPI = 110
fig, ax = plt.subplots(figsize=(1280 / DPI, 720 / DPI), dpi=DPI, facecolor='black')
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.axis('off')

# Basemap
ax.imshow(
    basemap,
    extent=bm_extent,          # [x0, x1, y0, y1] in Mercator
    origin='upper',
    aspect='auto',
    zorder=0
)

# Scatter artists
scatters = {}
for vtype, st in STYLE.items():
    scatters[vtype] = ax.scatter(
        [], [], s=st['size'], c=st['color'],
        marker=st['marker'], zorder=st['zorder'],
        edgecolors='white', linewidths=0.3, alpha=0.92
    )

trail_lines = []

# HUD — title
ax.text(0.5, 0.977,
        'SINGAPORE STRAIT — VESSEL TRAFFIC',
        transform=ax.transAxes, ha='center', va='top',
        fontsize=13, fontweight='bold', color='white', family='monospace',
        path_effects=[pe.withStroke(linewidth=2, foreground='black')], zorder=10)

# HUD — clock
clock_txt = ax.text(
    0.013, 0.977, '',
    transform=ax.transAxes, ha='left', va='top',
    fontsize=9, color='#a0ffcc', family='monospace',
    path_effects=[pe.withStroke(linewidth=2, foreground='black')], zorder=10)

# HUD — counts
count_txt = ax.text(
    0.987, 0.977, '',
    transform=ax.transAxes, ha='right', va='top',
    fontsize=8.5, color='white', family='monospace',
    path_effects=[pe.withStroke(linewidth=2, foreground='black')], zorder=10)

# Legend
legend_items = [
    mpatches.Patch(color='#00cfff', label='Eastbound'),
    mpatches.Patch(color='#ff6b35', label='Westbound'),
    mpatches.Patch(color='#ffe066', label='SW Approach'),
    mpatches.Patch(color='#a0ff80', label='Northbound'),
    mpatches.Patch(color='#ff80d5', label='Southbound'),
    mpatches.Patch(color='#ffdd00', label='Anchored'),
]
ax.legend(handles=legend_items, loc='lower left', fontsize=7.5,
          framealpha=0.45, facecolor='black', edgecolor='#445566',
          labelcolor='white', handlelength=1.2, borderpad=0.6, labelspacing=0.35)

# Scale bar (~10 km)
sb_x0, sb_y0 = transformer.transform(103.88, 1.165)
sb_x1, _     = transformer.transform(103.98, 1.165)
sb_y = sb_y0
tick = (ymax - ymin) * 0.008
ax.plot([sb_x0, sb_x1], [sb_y, sb_y], lw=2.5, color='white',
        solid_capstyle='butt', zorder=9)
ax.plot([sb_x0, sb_x0], [sb_y - tick, sb_y + tick], lw=2, color='white', zorder=9)
ax.plot([sb_x1, sb_x1], [sb_y - tick, sb_y + tick], lw=2, color='white', zorder=9)
ax.text((sb_x0 + sb_x1) / 2, sb_y - tick * 1.8, '~10 km',
        ha='center', va='top', fontsize=7.5, color='white', family='monospace',
        path_effects=[pe.withStroke(linewidth=1.5, foreground='black')], zorder=9)

# ═══════════════════════════════════════════════════════════════════════════════
# 8. UPDATE FUNCTION
# ═══════════════════════════════════════════════════════════════════════════════

def update(frame):
    global trail_lines
    for ln in trail_lines:
        ln.remove()
    trail_lines = []

    pts = {vt: [] for vt in STYLE}

    for i, v in enumerate(vessels):
        xs, ys = merc_coords[i]
        if np.isnan(xs[frame]):
            continue
        vtype = v['type']
        pts[vtype].append((xs[frame], ys[frame]))

        if vtype != 'anchored':
            start = max(0, frame - TRAIL)
            tx = xs[start:frame + 1]
            ty = ys[start:frame + 1]
            valid = ~np.isnan(tx)
            if valid.sum() > 1:
                alpha = max(0.08, 0.55 * (valid.sum() / TRAIL))
                ln, = ax.plot(tx[valid], ty[valid],
                              color=STYLE[vtype]['color'],
                              lw=0.9, alpha=alpha, zorder=2)
                trail_lines.append(ln)

    for vtype, sc in scatters.items():
        p = pts[vtype]
        sc.set_offsets(np.array(p) if p else np.empty((0, 2)))

    # Clock
    sim_h = SIM_START + (frame / N_FRAMES) * SIM_HOURS
    hh, mm = int(sim_h) % 24, int((sim_h % 1) * 60)
    clock_txt.set_text(f'SGT {hh:02d}:{mm:02d}')

    # Counts
    counts = {t: len(pts[t]) for t in STYLE}
    total  = sum(counts.values())
    count_txt.set_text(
        f'E:{counts["eastbound"]}  W:{counts["westbound"]}  '
        f'CR:{counts["crossing"] + counts["northbound"] + counts["southbound"]}  '
        f'A:{counts["anchored"]}  │  {total} vessels'
    )
    return [*scatters.values(), *trail_lines, clock_txt, count_txt]

# ═══════════════════════════════════════════════════════════════════════════════
# 9. RENDER TO MP4
# ═══════════════════════════════════════════════════════════════════════════════

OUTPUT_PATH = '/tmp/singapore_strait_vessels.mp4'

writer = FFMpegWriter(
    fps=FPS, codec='libx264', bitrate=4500,
    extra_args=['-pix_fmt', 'yuv420p', '-preset', 'fast',
                '-crf', '18', '-movflags', '+faststart']
)

print(f'\n🎬 Rendering {N_FRAMES} frames → {DURATION_S}s MP4 …')
t0 = time.time()

with writer.saving(fig, OUTPUT_PATH, dpi=DPI):
    for frame in tqdm(range(N_FRAMES), desc='Rendering', unit='frame'):
        update(frame)
        writer.grab_frame()

elapsed = time.time() - t0
size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f'\n✅  Done in {elapsed / 60:.1f} min  →  {size_mb:.1f} MB')
files.download(OUTPUT_PATH)

⬇️  Downloading basemap (Esri World Imagery via contextily)…
✅  Basemap saved  1280×768 px
Total frames: 4320  (180s × 24fps)  |  102 vessels
Vessels spawned: 102
⚙️  Pre-computing Mercator coordinates…
✅  Done

🎬 Rendering 4320 frames → 180s MP4 …


Rendering:   0%|          | 0/4320 [00:00<?, ?frame/s]


✅  Done in 14.3 min  →  10.0 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>